# Overview

Adapted Rossmann sales forecasting experiments using ARIMA, FB Prophet, and XGBoost with a log-transformed target. The original paper does not specify a particular normalization method, so this notebook uses `log1p(Sales)` consistently across all models.


# Model Architectures

# Reference-Based Exploration

This notebook adapts the reference experiment using a log-transformed target. It is not presented as an exact reproduction of an undocumented normalization method from the paper.


## Preprocessing

Data loading, merging, cleaning, cross-store daily aggregation, and chronological train-test splitting follow the reference setup. The target transformation is `Sales_log = log1p(Sales)`.


In [1]:
import sys
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

print('Imports successful!')

d:\Work-Env\ITEC\forecasting-medicine-public\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Imports successful!


In [2]:
PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'explore' or PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

train_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'rossmann', 'train.csv')
store_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'rossmann', 'store.csv')

df_train = pd.read_csv(train_path, low_memory=False, parse_dates=['Date'])
df_store = pd.read_csv(store_path, low_memory=False)
df_merged = pd.merge(df_train, df_store, on='Store', how='left')

df_cleaned = df_merged[(df_merged['Open'] != 0) & (df_merged['Sales'] > 0)].copy()

df_daily = df_cleaned.groupby('Date').agg({
    'Sales': 'sum',
    'Promo': 'mean',
    'SchoolHoliday': 'mean'
}).reset_index().sort_values('Date').reset_index(drop=True)

df_daily['DayOfWeek'] = df_daily['Date'].dt.dayofweek
df_daily['year'] = df_daily['Date'].dt.year
df_daily['month'] = df_daily['Date'].dt.month
df_daily['day'] = df_daily['Date'].dt.day

print(f'Cross-store daily dataset size: {df_daily.shape}')
print(f'Rentang tanggal: {df_daily["Date"].min()} s.d {df_daily["Date"].max()}')

Cross-store daily dataset size: (942, 8)
Rentang tanggal: 2013-01-01 00:00:00 s.d 2015-07-31 00:00:00


In [3]:
split_idx = int(len(df_daily) * 0.8)
train_df = df_daily.iloc[:split_idx].copy()
test_df = df_daily.iloc[split_idx:].copy()

train_df["Sales_log"] = np.log1p(train_df["Sales"])
test_df["Sales_log"] = np.log1p(test_df["Sales"])

print(f"Train set: {train_df.shape[0]} days")
print(f"Test set: {test_df.shape[0]} days")
print(f"Training target: log1p(Sales)")


Train set: 753 days
Test set: 189 days
Training target: log1p(Sales)


## Modeling

All models use `Sales_log` during training. Predictions are evaluated on both log-transformed and original sales scales.


### Model 1: ARIMA (Log-Transformed)


In [4]:
train_sales_log = train_df["Sales_log"].to_numpy()
test_sales_log = test_df["Sales_log"].to_numpy()
test_sales_original = test_df["Sales"].to_numpy()

arima_model = ARIMA(train_sales_log, order=(1, 1, 1)).fit()
preds_arima_log = arima_model.forecast(steps=len(test_sales_log))
preds_arima_original = np.expm1(preds_arima_log)

rmse_arima_log = np.sqrt(mean_squared_error(test_sales_log, preds_arima_log))
mse_arima_log = mean_squared_error(test_sales_log, preds_arima_log)
mae_arima_log = mean_absolute_error(test_sales_log, preds_arima_log)
r2_arima_log = r2_score(test_sales_log, preds_arima_log)
rmse_arima_original = np.sqrt(mean_squared_error(test_sales_original, preds_arima_original))
mse_arima_original = mean_squared_error(test_sales_original, preds_arima_original)
mae_arima_original = mean_absolute_error(test_sales_original, preds_arima_original)
r2_arima_original = r2_score(test_sales_original, preds_arima_original)


### Model 2: FB Prophet (Log-Transformed)


In [5]:
train_prophet_log = train_df[["Date", "Sales_log"]].rename(columns={"Date": "ds", "Sales_log": "y"})
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
prophet_model.fit(train_prophet_log)

future = pd.DataFrame({"ds": test_df["Date"]})
preds_prophet_log = prophet_model.predict(future)["yhat"].to_numpy()
preds_prophet_original = np.expm1(preds_prophet_log)

rmse_prophet_log = np.sqrt(mean_squared_error(test_sales_log, preds_prophet_log))
mse_prophet_log = mean_squared_error(test_sales_log, preds_prophet_log)
mae_prophet_log = mean_absolute_error(test_sales_log, preds_prophet_log)
r2_prophet_log = r2_score(test_sales_log, preds_prophet_log)
rmse_prophet_original = np.sqrt(mean_squared_error(test_sales_original, preds_prophet_original))
mse_prophet_original = mean_squared_error(test_sales_original, preds_prophet_original)
mae_prophet_original = mean_absolute_error(test_sales_original, preds_prophet_original)
r2_prophet_original = r2_score(test_sales_original, preds_prophet_original)


13:49:56 - cmdstanpy - INFO - Chain [1] start processing
13:49:57 - cmdstanpy - INFO - Chain [1] done processing


### Model 3: XGBoost (Log-Transformed)


In [6]:
features = ["DayOfWeek", "Promo", "SchoolHoliday", "year", "month", "day"]
X_train = train_df[features]
X_test = test_df[features]

xgb_model = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, train_df["Sales_log"])
preds_xgb_log = xgb_model.predict(X_test)
preds_xgb_original = np.expm1(preds_xgb_log)

rmse_xgb_log = np.sqrt(mean_squared_error(test_sales_log, preds_xgb_log))
mse_xgb_log = mean_squared_error(test_sales_log, preds_xgb_log)
mae_xgb_log = mean_absolute_error(test_sales_log, preds_xgb_log)
r2_xgb_log = r2_score(test_sales_log, preds_xgb_log)
rmse_xgb_original = np.sqrt(mean_squared_error(test_sales_original, preds_xgb_original))
mse_xgb_original = mean_squared_error(test_sales_original, preds_xgb_original)
mae_xgb_original = mean_absolute_error(test_sales_original, preds_xgb_original)
r2_xgb_original = r2_score(test_sales_original, preds_xgb_original)


### Log-Transformed Results


In [7]:
df_log_results = pd.DataFrame([
    {
        "Model": "ARIMA",
        "RMSE (Log)": rmse_arima_log, "MSE (Log)": mse_arima_log, "MAE (Log)": mae_arima_log, "R2 (Log)": r2_arima_log,
        "RMSE (Original)": rmse_arima_original, "MSE (Original)": mse_arima_original, "MAE (Original)": mae_arima_original, "R2 (Original)": r2_arima_original,
    },
    {
        "Model": "FB Prophet",
        "RMSE (Log)": rmse_prophet_log, "MSE (Log)": mse_prophet_log, "MAE (Log)": mae_prophet_log, "R2 (Log)": r2_prophet_log,
        "RMSE (Original)": rmse_prophet_original, "MSE (Original)": mse_prophet_original, "MAE (Original)": mae_prophet_original, "R2 (Original)": r2_prophet_original,
    },
    {
        "Model": "XGBoost",
        "RMSE (Log)": rmse_xgb_log, "MSE (Log)": mse_xgb_log, "MAE (Log)": mae_xgb_log, "R2 (Log)": r2_xgb_log,
        "RMSE (Original)": rmse_xgb_original, "MSE (Original)": mse_xgb_original, "MAE (Original)": mae_xgb_original, "R2 (Original)": r2_xgb_original,
    },
])
df_log_results


,Model,RMSE (Log),MSE (Log),MAE (Log),R2 (Log),RMSE (Original),MSE (Original),MAE (Original),R2 (Original)
0,ARIMA,1.309429,1.714606,1.026861,-0.004490,4.167510e+06,1.736814e+13,3.898402e+06,-0.665937
1,FB Prophet,0.587415,0.345057,0.333999,0.797851,2.386493e+06,5.695351e+12,1.754654e+06,0.453707
2,XGBoost,0.443693,0.196864,0.183355,0.884669,1.589091e+06,2.525211e+12,9.083837e+05,0.757784


## Summary

This notebook uses one target transformation consistently: `Sales_log = log1p(Sales)`. Log-scale metrics describe model fit during training, while original-scale metrics are computed after inverse transformation with `expm1`.
